# Project 06 — Stock Market Price Forecasting
**Time Series Analysis, Technical Indicators, ARIMA, SARIMA and Prophet**

| Item | Detail |
|---|---|
| Domain | Financial time series / quantitative analysis |
| Dataset | `stock_price_data.csv` — 1,566 daily OHLCV records (Jan 2019 – Dec 2024) |
| Tools | Python, pandas, statsmodels, Prophet, matplotlib, seaborn |
| Level | Intermediate to advanced |

**Goals**
1. Explore the price, volume and returns, and compute technical indicators.
2. Decompose the series and test for stationarity (ADF and KPSS).
3. Fit Naive, Drift, ARIMA, SARIMA and Prophet models.
4. Evaluate with MAE, RMSE, MAPE and walk-forward (one-step-ahead) validation.
5. Check residuals, produce a 95% forecast interval, and be honest about limits.

> ⚠️ The data is **synthetic** (generated for teaching). Never use it for real trading decisions.

## 0. Setup
Run the next cell once if a library is missing (Google Colab already has them all):

In [ ]:
# !pip install statsmodels prophet seaborn openpyxl

In [ ]:
import os, warnings, logging, itertools
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:                       # `display` exists in Jupyter; fall back to print elsewhere
    display
except NameError:
    display = print

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 100,
                     "axes.titleweight": "bold", "axes.spines.top": False,
                     "axes.spines.right": False})
BLUE, ORANGE, GREEN, RED, GREY, PURPLE = "#1f77b4", "#ff7f0e", "#2a9d8f", "#e76f51", "#8d99ae", "#7b2cbf"

### Helper functions
Two small statistics helpers written from scratch, so you can see what they do:
* **ACF** (autocorrelation) – how strongly today's value is related to the value *k* days ago.
* **Ljung-Box test** – tests whether a whole group of autocorrelations is zero (small p-value ⇒ there is structure left).

In [ ]:
def acf_values(x, nlags=30):
    """Sample autocorrelation for lags 0..nlags."""
    x = np.asarray(x, float)
    x = x - x.mean()
    denom = (x ** 2).sum()
    return np.array([(x[k:] * x[:len(x) - k]).sum() / denom for k in range(nlags + 1)])


def ljung_box(x, lags=10, dof_adj=0):
    """Ljung-Box Q statistic and p-value. dof_adj = number of fitted ARMA parameters."""
    x = np.asarray(x, float)
    n = len(x)
    r = acf_values(x, lags)
    q = n * (n + 2) * np.sum(r[1:] ** 2 / (n - np.arange(1, lags + 1)))
    return q, stats.chi2.sf(q, max(lags - dof_adj, 1))


def plot_acf_bars(ax, x, nlags=30, title=""):
    r = acf_values(x, nlags)[1:]
    band = 1.96 / np.sqrt(len(x))
    ax.bar(range(1, nlags + 1), r, color=BLUE, width=0.6)
    ax.axhline(band, color=RED, ls="--", lw=1)
    ax.axhline(-band, color=RED, ls="--", lw=1)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("Lag (days)")


def metrics(actual, pred):
    """MAE, RMSE, MAPE (%) on the price scale."""
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    return {"MAE": np.mean(np.abs(a - p)),
            "RMSE": np.sqrt(np.mean((a - p) ** 2)),
            "MAPE (%)": np.mean(np.abs((a - p) / a)) * 100}

## 1. Data audit
Load the data, parse the dates, check for missing values, duplicates, gaps and impossible prices.

In [ ]:
CSV, XLSX = "stock_price_data.csv", "stock_price_data.xlsx"
if os.path.exists(CSV):
    df = pd.read_csv(CSV, parse_dates=["Date"])
else:
    df = pd.read_excel(XLSX, parse_dates=["Date"])

df = df.sort_values("Date").set_index("Date")
df = df.asfreq("B")                      # business-day frequency (no gaps in this dataset)
print("Shape:", df.shape, "| from", df.index.min().date(), "to", df.index.max().date())
display(df.head())
display(df.describe().T.round(2))

In [ ]:
print("Missing values per column:\n", df.isna().sum().to_string(), "\n")
print("Duplicate dates          :", df.index.duplicated().sum())
all_bdays = pd.bdate_range(df.index.min(), df.index.max())
print("Missing business days    :", len(all_bdays.difference(df.dropna().index)))
bad = df[(df.High < df[["Open", "Close"]].max(axis=1)) | (df.Low > df[["Open", "Close"]].min(axis=1))]
print("Rows with inconsistent OHLC:", len(bad))
print("Adj_Close identical to Close:", np.allclose(df.Adj_Close, df.Close))

**Reading the audit:** no missing values, no duplicate dates, every business day present and the High/Low
prices are consistent with Open/Close. The data is clean, so we can analyse it directly.

## 2. Market (technical) analysis
### 2.1 Price with moving averages
A *moving average* (MA) smooths the price. When the short MA crosses above the long MA traders call it a bullish signal.

In [ ]:
for w in (20, 50, 200):
    df[f"MA{w}"] = df.Close.rolling(w).mean()

fig, ax = plt.subplots()
ax.plot(df.Close, color=GREY, lw=1, label="Close")
ax.plot(df.MA20, color=BLUE, lw=1.4, label="MA 20")
ax.plot(df.MA50, color=ORANGE, lw=1.4, label="MA 50")
ax.plot(df.MA200, color=RED, lw=1.8, label="MA 200")
ax.set_title("Close price with 20/50/200-day moving averages")
ax.set_ylabel("Price")
ax.legend(ncol=4)
plt.tight_layout(); plt.show()

### 2.2 Candlestick chart (last 90 trading days)
Each candle shows Open→Close (the body) and Low→High (the wick). Green = closed higher than it opened.

In [ ]:
def candlestick(ax, data):
    x = np.arange(len(data))
    up = (data.Close >= data.Open).values
    colors = np.where(up, GREEN, RED)
    ax.vlines(x, data.Low, data.High, colors=colors, lw=1)
    body = (data.Close - data.Open).abs().clip(lower=0.05)
    ax.bar(x, body, bottom=np.minimum(data.Open, data.Close), color=colors, width=0.7)
    step = max(len(data) // 9, 1)
    ax.set_xticks(x[::step])
    ax.set_xticklabels([d.strftime("%d %b %y") for d in data.index[::step]])

fig, ax = plt.subplots()
candlestick(ax, df.iloc[-90:])
ax.set_title("Candlestick chart – last 90 trading days")
ax.set_ylabel("Price")
plt.tight_layout(); plt.show()

### 2.3 Bollinger Bands, RSI and MACD
* **Bollinger Bands** – 20-day MA ± 2 standard deviations. Price touching the upper band = "stretched" upward.
* **RSI (14)** – momentum from 0–100. Above 70 = *overbought*, below 30 = *oversold*.
* **MACD** – EMA(12) − EMA(26) with a 9-day signal line. The histogram is the gap between the two.

In [ ]:
# Bollinger Bands
df["BB_mid"] = df.Close.rolling(20).mean()
bb_std = df.Close.rolling(20).std()
df["BB_up"], df["BB_low"] = df.BB_mid + 2 * bb_std, df.BB_mid - 2 * bb_std

# RSI (Wilder smoothing)
delta = df.Close.diff()
gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
avg_loss = loss.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
df["RSI"] = 100 - 100 / (1 + avg_gain / avg_loss)

# MACD
ema12 = df.Close.ewm(span=12, adjust=False).mean()
ema26 = df.Close.ewm(span=26, adjust=False).mean()
df["MACD"] = ema12 - ema26
df["MACD_signal"] = df.MACD.ewm(span=9, adjust=False).mean()
df["MACD_hist"] = df.MACD - df.MACD_signal

view = df.iloc[-300:]
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [3, 1.3, 1.5]})
axes[0].plot(view.Close, color="black", lw=1.2, label="Close")
axes[0].plot(view.BB_mid, color=ORANGE, lw=1, label="20-day MA")
axes[0].fill_between(view.index, view.BB_low, view.BB_up, color=BLUE, alpha=0.15, label="Bollinger band (±2σ)")
axes[0].set_title("Price and Bollinger Bands (last 300 days)"); axes[0].legend(loc="upper left", ncol=3)

axes[1].plot(view.RSI, color=PURPLE)
axes[1].axhline(70, color=RED, ls="--", lw=1); axes[1].axhline(30, color=GREEN, ls="--", lw=1)
axes[1].fill_between(view.index, 30, 70, color=GREY, alpha=0.1)
axes[1].set_ylim(0, 100); axes[1].set_title("RSI (14)")

axes[2].plot(view.MACD, color=BLUE, label="MACD"); axes[2].plot(view.MACD_signal, color=ORANGE, label="Signal")
axes[2].bar(view.index, view.MACD_hist, color=np.where(view.MACD_hist >= 0, GREEN, RED), width=1.0, alpha=0.6)
axes[2].axhline(0, color="black", lw=0.8); axes[2].set_title("MACD (12, 26, 9)"); axes[2].legend(loc="upper left", ncol=2)
plt.tight_layout(); plt.show()

print(f"Days RSI > 70 (overbought): {(df.RSI > 70).sum()} | Days RSI < 30 (oversold): {(df.RSI < 30).sum()}")

### 2.4 Drawdown
*Drawdown* = how far the price is below its previous all-time high. It measures the pain an investor would have felt.

In [ ]:
running_max = df.Close.cummax()
df["Drawdown"] = df.Close / running_max - 1
max_dd, max_dd_date = df.Drawdown.min(), df.Drawdown.idxmin()
peak_date = df.Close.loc[:max_dd_date].idxmax()
print(f"Maximum drawdown: {max_dd:.1%} (peak on {peak_date.date()}, trough on {max_dd_date.date()})")

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(df.index, df.Drawdown * 100, 0, color=RED, alpha=0.5)
ax.set_title("Drawdown from running peak"); ax.set_ylabel("Drawdown (%)")
plt.tight_layout(); plt.show()

### 2.5 Monthly return heatmap and volume

In [ ]:
monthly = df.Close.resample("ME").last().pct_change().dropna() * 100
heat = (monthly.to_frame("ret").assign(Year=monthly.index.year, Month=monthly.index.month)
        .pivot(index="Year", columns="Month", values="ret"))
heat.columns = [pd.Timestamp(2000, m, 1).strftime("%b") for m in heat.columns]

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn", center=0, linewidths=0.5, cbar_kws={"label": "Monthly return (%)"}, ax=ax)
ax.set_title("Monthly returns (%)")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(df.Volume, color=GREY, lw=0.6)
axes[0].plot(df.Volume.rolling(20).mean(), color=BLUE, lw=1.5, label="20-day average")
axes[0].set_title("Daily volume"); axes[0].legend()
axes[1].scatter(df.Close.pct_change().abs() * 100, df.Volume, s=6, alpha=0.4, color=PURPLE)
axes[1].set_xlabel("|daily return| (%)"); axes[1].set_ylabel("Volume"); axes[1].set_title("Volume rises on volatile days")
plt.tight_layout(); plt.show()

## 3. Statistical analysis of returns
Models work better on **returns** than on raw prices. We use *log returns*: `log(P_t) − log(P_{t−1})`,
which are additive over time and approximately equal to the % change.

In [ ]:
df["Ret"] = df.Close.pct_change()
df["LogRet"] = np.log(df.Close).diff()
df["AbsRet"] = df.LogRet.abs()
r = df.LogRet.dropna()

jb_stat, jb_p = stats.jarque_bera(r)
sh_stat, sh_p = stats.shapiro(r)
summary = pd.Series({
    "Mean daily log-return (%)": r.mean() * 100,
    "Daily volatility (%)": r.std() * 100,
    "Annualised volatility (%)": r.std() * np.sqrt(252) * 100,
    "Skewness": stats.skew(r),
    "Excess kurtosis": stats.kurtosis(r),
    "Jarque-Bera p-value": jb_p,
    "Shapiro-Wilk p-value": sh_p,
})
display(summary.round(4).to_frame("Value"))
print("Normality is", "REJECTED (returns are not normal – fat tails)" if jb_p < 0.05 else "not rejected")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(r * 100, color=BLUE, lw=0.7); axes[0].set_title("Daily log-returns (%)")
axes[1].hist(r * 100, bins=60, density=True, color=BLUE, alpha=0.6, label="Returns")
xs = np.linspace(r.min() * 100, r.max() * 100, 300)
axes[1].plot(xs, stats.norm.pdf(xs, r.mean() * 100, r.std() * 100), color=RED, label="Normal curve")
axes[1].set_title("Return distribution"); axes[1].legend()
stats.probplot(r, dist="norm", plot=axes[2])
axes[2].get_lines()[0].set(markerfacecolor=BLUE, markeredgecolor=BLUE, markersize=3)
axes[2].set_title("Q-Q plot vs normal")
plt.tight_layout(); plt.show()

**Fat tails:** the histogram is more peaked with heavier tails than the red normal curve, and the Q-Q plot bends away from the line at both ends.
This means large moves happen much more often than a normal distribution predicts — important for risk.

### 3.1 Volatility clustering
Big moves tend to follow big moves. Returns themselves are almost unpredictable (ACF ≈ 0), but **squared / absolute returns are not** — the signature of GARCH-type volatility.

In [ ]:
df["Vol21"] = df.LogRet.rolling(21).std() * np.sqrt(252) * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(df.Vol21, color=ORANGE); axes[0].set_title("Rolling 21-day annualised volatility (%)")
plot_acf_bars(axes[1], r, 30, "ACF of returns")
plot_acf_bars(axes[2], r ** 2, 30, "ACF of squared returns")
plt.tight_layout(); plt.show()

rows = []
for name, series in [("Returns", r), ("Squared returns", r ** 2)]:
    q, p = ljung_box(series, lags=10)
    rows.append({"Series": name, "Ljung-Box Q (10 lags)": q, "p-value": p,
                 "Conclusion": "autocorrelated" if p < 0.05 else "no autocorrelation"})
display(pd.DataFrame(rows).round(4))

print("Five largest absolute daily moves (likely the injected shock events):")
display((df.LogRet.abs().nlargest(5) * 100).round(2).to_frame("|log-return| %"))

### 3.2 Correlation

In [ ]:
corr_cols = ["Close", "Volume", "Ret", "AbsRet", "Vol21"]
corr = df[corr_cols].dropna().corr()
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout(); plt.show()
print(f"Correlation of today's return with yesterday's: {df.Ret.corr(df.Ret.shift(1)):.3f}  (≈ 0 ⇒ little linear predictability)")

## 4. Time-series structure
### 4.1 STL decomposition
STL splits the series into **Trend + Seasonal + Residual**. We use the *log* price so that the components are additive
even though price swings grow with the price level. Period = 252 trading days ≈ one year.

In [ ]:
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

y = np.log(df["Close"])
stl = STL(y, period=252, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(12, 8); fig.suptitle("STL decomposition of log(Close)", y=1.0, fontweight="bold")
plt.tight_layout(); plt.show()

var_r = np.var(stl.resid)
Ft = max(0, 1 - var_r / np.var(stl.trend + stl.resid))
Fs = max(0, 1 - var_r / np.var(stl.seasonal + stl.resid))
print(f"Strength of trend       : {Ft:.2f}  (1 = very strong)")
print(f"Strength of seasonality : {Fs:.2f}  (close to 0 = weak / mild)")

### 4.2 Stationarity tests (ADF and KPSS)
Most models (ARIMA) need a *stationary* series: constant mean and variance over time.
* **ADF** – null hypothesis: the series has a unit root (non-stationary). **p < 0.05 ⇒ stationary.**
* **KPSS** – null hypothesis: the series *is* stationary. **p > 0.05 ⇒ stationary.**

If both agree, we can trust the verdict.

In [ ]:
def stationarity_report(s, name):
    s = s.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    adf_ok, kpss_ok = adf_p < 0.05, kpss_p > 0.05
    verdict = ("Stationary" if adf_ok and kpss_ok else
               "Non-stationary" if (not adf_ok) and (not kpss_ok) else "Inconclusive")
    return {"Series": name, "ADF p-value": adf_p, "KPSS p-value": kpss_p, "Verdict": verdict}

tests = pd.DataFrame([
    stationarity_report(df.Close, "Close"),
    stationarity_report(y, "log(Close)"),
    stationarity_report(df.Close.diff(), "Close, 1st difference"),
    stationarity_report(y.diff(), "log(Close), 1st difference (= log-returns)"),
])
display(tests.round(4))
print("→ Prices are non-stationary; one difference (d = 1) makes the log series stationary.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y.diff().dropna(), lags=40, ax=axes[0]); axes[0].set_title("ACF of differenced log price")
plot_pacf(y.diff().dropna(), lags=40, ax=axes[1], method="ywm"); axes[1].set_title("PACF of differenced log price")
plt.tight_layout(); plt.show()

**Reading the ACF/PACF:** after differencing, almost every bar is inside the blue band — there is very little
linear structure left to exploit. Expect ARIMA to find only weak AR/MA terms and to be close to a random walk with drift.

## 5. Modelling
**Set-up:** we model the **log price** and hold out the **last 60 trading days** as a test set
(train = everything before). All errors are reported back on the **price scale**.

| Model | Idea |
|---|---|
| Naive | tomorrow = today (the benchmark everything must beat) |
| Drift | Naive + the average historical daily change |
| ARIMA(p,1,q) | order chosen by AIC, with a drift term |
| SARIMA | ARIMA + a seasonal part (5-day trading week) |
| ARIMA + Fourier | ARIMA with yearly seasonality supplied as sine/cosine regressors |
| Prophet | additive trend + yearly seasonality model from Meta |

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

H = 60
train, test = y.iloc[:-H], y.iloc[-H:]
actual = df["Close"].iloc[-H:]
print(f"Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} days)")
print(f"Test : {test.index[0].date()} → {test.index[-1].date()} ({len(test)} days)")

forecasts, intervals = {}, {}      # price-scale forecasts and (lower, upper) 95% bands

### 5.1 Baselines: Naive and Drift

In [ ]:
steps = np.arange(1, H + 1)
forecasts["Naive"] = pd.Series(np.exp(train.iloc[-1]) * np.ones(H), index=test.index)

slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
forecasts["Drift"] = pd.Series(np.exp(train.iloc[-1] + slope * steps), index=test.index)
print(f"Drift = {slope * 100:.4f}% per trading day")

### 5.2 ARIMA — choose (p, q) by AIC
We already know **d = 1**. We try every p, q in 0…3 and keep the one with the lowest AIC
(AIC = fit quality with a penalty for extra parameters; lower is better).

In [ ]:
rows = []
for p, q in itertools.product(range(4), range(4)):
    try:
        res = ARIMA(train, order=(p, 1, q), trend="t").fit()
        rows.append({"p": p, "d": 1, "q": q, "AIC": res.aic, "BIC": res.bic})
    except Exception as e:
        print(f"ARIMA({p},1,{q}) failed: {e}")
order_table = pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)
display(order_table.head(6).round(2))

best_p, best_d, best_q = (int(v) for v in order_table.loc[0, ["p", "d", "q"]])
best_order = (best_p, best_d, best_q)
print("Selected order:", best_order)

In [ ]:
arima_res = ARIMA(train, order=best_order, trend="t").fit()
print(arima_res.summary().tables[1])

fc = arima_res.get_forecast(H)
ci = np.exp(fc.conf_int(alpha=0.05)).values
forecasts["ARIMA"] = pd.Series(np.exp(fc.predicted_mean.values), index=test.index)
intervals["ARIMA"] = (pd.Series(ci[:, 0], index=test.index), pd.Series(ci[:, 1], index=test.index))

### 5.3 SARIMA and ARIMA + Fourier (seasonality)
A full yearly seasonal period (s = 252) is far too large for a classical SARIMA, so we test two practical routes:
* **SARIMA** with a 5-day (weekly trading cycle) seasonal part: `(1,0,1,5)`.
* **ARIMA + Fourier terms** – yearly seasonality fed in as sin/cos regressors (the standard trick for long periods).

In [ ]:
sarima_res = ARIMA(train, order=best_order, seasonal_order=(1, 0, 1, 5), trend="t").fit()
fc = sarima_res.get_forecast(H)
ci = np.exp(fc.conf_int(alpha=0.05)).values
forecasts["SARIMA"] = pd.Series(np.exp(fc.predicted_mean.values), index=test.index)
intervals["SARIMA"] = (pd.Series(ci[:, 0], index=test.index), pd.Series(ci[:, 1], index=test.index))
print(f"AIC  ARIMA {arima_res.aic:.1f}   vs   SARIMA {sarima_res.aic:.1f}  (lower is better)")

def fourier_terms(index, K=2, period=365.25):
    t = np.asarray(index.dayofyear) / period
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(2 * np.pi * k * t)
        cols[f"cos{k}"] = np.cos(2 * np.pi * k * t)
    return pd.DataFrame(cols, index=index)

X_all = fourier_terms(y.index)
X_train, X_test = X_all.iloc[:-H], X_all.iloc[-H:]
fourier_res = ARIMA(train, exog=X_train, order=best_order, trend="t").fit()
fc = fourier_res.get_forecast(H, exog=X_test)
ci = np.exp(fc.conf_int(alpha=0.05)).values
forecasts["ARIMA + Fourier"] = pd.Series(np.exp(fc.predicted_mean.values), index=test.index)
intervals["ARIMA + Fourier"] = (pd.Series(ci[:, 0], index=test.index), pd.Series(ci[:, 1], index=test.index))
print(f"AIC  ARIMA {arima_res.aic:.1f}   vs   ARIMA+Fourier {fourier_res.aic:.1f}")

### 5.4 Prophet
Prophet fits `trend + yearly seasonality`. Stock data has no weekly pattern here (business days only), so weekly seasonality is switched off.

In [ ]:
from prophet import Prophet

def make_prophet():
    return Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                   changepoint_prior_scale=0.05, interval_width=0.95)

def prophet_frame(series):
    return pd.DataFrame({"ds": series.index, "y": series.values})

m = make_prophet()
m.fit(prophet_frame(train))
pp = m.predict(pd.DataFrame({"ds": test.index}))
forecasts["Prophet"] = pd.Series(np.exp(pp["yhat"].values), index=test.index)
intervals["Prophet"] = (pd.Series(np.exp(pp["yhat_lower"].values), index=test.index),
                        pd.Series(np.exp(pp["yhat_upper"].values), index=test.index))

fig = m.plot_components(m.predict(prophet_frame(train)))   # fitted trend and yearly pattern
fig.set_size_inches(10, 6)
plt.tight_layout(); plt.show()

### 5.5 All 60-day forecasts vs reality

In [ ]:
palette = {"Naive": GREY, "Drift": GREEN, "ARIMA": BLUE, "SARIMA": ORANGE, "ARIMA + Fourier": PURPLE, "Prophet": RED}
fig, ax = plt.subplots(figsize=(13, 5.5))
hist = df.Close.iloc[-(H + 120):-H]
ax.plot(hist, color="black", lw=1.2, label="History (train)")
ax.plot(actual, color="black", lw=2.4, label="Actual (test)")
for name, f in forecasts.items():
    ax.plot(f, color=palette[name], lw=1.6, ls="--", label=name)
ax.axvline(test.index[0], color=GREY, ls=":")
ax.set_title("60-day-ahead forecasts vs actual closing price"); ax.set_ylabel("Price")
ax.legend(ncol=4)
plt.tight_layout(); plt.show()

## 6. Evaluation
### 6.1 Error metrics on the 60-day hold-out
* **MAE** – average absolute error (in price units).
* **RMSE** – like MAE but punishes large misses more.
* **MAPE** – average error as a % of the actual price.

A model is only useful if it **beats Naive**.

In [ ]:
results = pd.DataFrame({name: metrics(actual, f) for name, f in forecasts.items()}).T
results["RMSE vs Naive (%)"] = (results["RMSE"] / results.loc["Naive", "RMSE"] - 1) * 100
results["Beats Naive?"] = [
    "—" if n == "Naive" else ("Yes" if (results.loc[n, "MAE"] < results.loc["Naive", "MAE"]
                                         and results.loc[n, "RMSE"] < results.loc["Naive", "RMSE"]) else "No")
    for n in results.index]
results = results.sort_values("RMSE")
display(results.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["MAE", "RMSE", "MAPE (%)"]):
    results[col].sort_values().plot.barh(ax=ax, color=[palette[i] for i in results[col].sort_values().index])
    ax.set_title(col)
plt.tight_layout(); plt.show()

### 6.2 Residual diagnostics (ARIMA)
Good residuals look like **white noise**: no pattern, no autocorrelation, roughly normal.
We test both the residuals and the *squared* residuals (to detect volatility clustering).

In [ ]:
resid = arima_res.resid.iloc[1:]          # first value is an initialisation artefact
npar = best_p + best_q
q1, p1 = ljung_box(resid, lags=20, dof_adj=npar)
q2, p2 = ljung_box(resid ** 2, lags=20)
print(f"Ljung-Box on residuals          : Q={q1:.1f}, p={p1:.4f} → {'structure remains' if p1 < 0.05 else 'looks like white noise'}")
print(f"Ljung-Box on squared residuals  : Q={q2:.1f}, p={p2:.4f} → {'volatility clustering (ARCH effect)' if p2 < 0.05 else 'no ARCH effect'}")
print(f"Residual skew {stats.skew(resid):.2f}, excess kurtosis {stats.kurtosis(resid):.2f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(resid, color=BLUE, lw=0.7); axes[0, 0].axhline(0, color="black"); axes[0, 0].set_title("Residuals over time")
axes[0, 1].hist(resid, bins=50, density=True, color=BLUE, alpha=0.6)
xs = np.linspace(resid.min(), resid.max(), 300)
axes[0, 1].plot(xs, stats.norm.pdf(xs, resid.mean(), resid.std()), color=RED); axes[0, 1].set_title("Residual histogram")
plot_acf_bars(axes[1, 0], resid, 30, "ACF of residuals")
plot_acf_bars(axes[1, 1], resid ** 2, 30, "ACF of squared residuals")
plt.tight_layout(); plt.show()

### 6.3 Is the 95% interval trustworthy?
We count how many of the 60 actual closing prices fell inside each model's 95% band (it *should* be about 95%).

In [ ]:
cov = {name: float(((actual >= lo) & (actual <= hi)).mean() * 100) for name, (lo, hi) in intervals.items()}
display(pd.Series(cov).round(1).to_frame("Actual inside 95% band (%)"))

### 6.4 Walk-forward (one-step-ahead) validation
The 60-day forecast above is a *hard* test (predicting up to 3 months blind). In practice you forecast **tomorrow**
using everything known up to **today**. Walk-forward does this for each of the last 60 days:
* Naive / Drift: recomputed from the data available the day before.
* ARIMA / SARIMA / Fourier: parameters stay fixed, but the model state is updated with each new observation (`apply(..., refit=False)`).
* Prophet: it cannot update daily, so it is **refit every 10 days** and predicts the next 10 days.

In [ ]:
n = len(y)
n_tr = n - H
wf = {}

wf["Naive"] = pd.Series(np.exp(y.shift(1).iloc[-H:].values), index=test.index)

pos = np.arange(n_tr, n)
slope_t = (y.iloc[pos - 1].values - y.iloc[0]) / (pos - 1)
wf["Drift"] = pd.Series(np.exp(y.iloc[pos - 1].values + slope_t), index=test.index)

def one_step(fitted, endog, exog=None):
    ext = fitted.apply(endog, exog=exog, refit=False)
    pred = ext.get_prediction(start=n_tr, end=n - 1).predicted_mean
    return pd.Series(np.exp(np.asarray(pred)), index=test.index)

wf["ARIMA"] = one_step(arima_res, y)
wf["SARIMA"] = one_step(sarima_res, y)
wf["ARIMA + Fourier"] = one_step(fourier_res, y, exog=X_all)

block, parts = 10, []
for s in range(0, H, block):
    cut = n_tr + s
    mp = make_prophet()
    mp.fit(prophet_frame(y.iloc[:cut]))
    parts.append(mp.predict(pd.DataFrame({"ds": y.index[cut:cut + block]}))["yhat"].values)
wf["Prophet"] = pd.Series(np.exp(np.concatenate(parts)), index=test.index)

wf_results = pd.DataFrame({k: metrics(actual, v) for k, v in wf.items()}).T
wf_results["RMSE vs Naive (%)"] = (wf_results["RMSE"] / wf_results.loc["Naive", "RMSE"] - 1) * 100
wf_results = wf_results.sort_values("RMSE")
display(wf_results.round(3))

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(actual, color="black", lw=2.2, label="Actual")
for name in ["Naive", "ARIMA", "Prophet"]:
    ax.plot(wf[name], color=palette[name], lw=1.4, ls="--", label=name)
ax.set_title("Walk-forward one-step-ahead predictions (last 60 days)"); ax.legend(ncol=4)
plt.tight_layout(); plt.show()

## 7. Final forecast — next 30 trading days with 95% interval
We now **refit on the full history** and forecast beyond the last date. The ARIMA order selected above is reused;
Prophet is refit from scratch.

In [ ]:
FUT = 30
future_idx = pd.bdate_range(df.index[-1] + pd.offsets.BDay(1), periods=FUT)

full_arima = ARIMA(y, order=best_order, trend="t").fit()
fa = full_arima.get_forecast(FUT)
a_mean = np.exp(fa.predicted_mean.values)
a_ci = np.exp(fa.conf_int(alpha=0.05)).values

pm = make_prophet()
pm.fit(prophet_frame(y))
pf = pm.predict(pd.DataFrame({"ds": future_idx}))

future = pd.DataFrame({
    "ARIMA": a_mean, "ARIMA_lower95": a_ci[:, 0], "ARIMA_upper95": a_ci[:, 1],
    "Prophet": np.exp(pf["yhat"].values), "Prophet_lower95": np.exp(pf["yhat_lower"].values),
    "Prophet_upper95": np.exp(pf["yhat_upper"].values)}, index=future_idx)
future.index.name = "Date"
display(future.round(2).iloc[[0, 4, 9, 14, 19, 24, 29]])
future.round(4).to_csv("forecast_next_30_days.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
hist = df.Close.iloc[-250:]
for ax, name, mean, lo, hi, c in [
        (axes[0], "ARIMA", a_mean, a_ci[:, 0], a_ci[:, 1], BLUE),
        (axes[1], "Prophet", future.Prophet.values, future.Prophet_lower95.values, future.Prophet_upper95.values, RED)]:
    ax.plot(hist, color="black", lw=1.2, label="History")
    ax.plot(future_idx, mean, color=c, lw=2, label=f"{name} forecast")
    ax.fill_between(future_idx, lo, hi, color=c, alpha=0.2, label="95% interval")
    ax.set_title(f"{name}: next {FUT} trading days"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()
print("Saved: forecast_next_30_days.csv")

## 8. Conclusions
The cell below writes the findings from the numbers computed above, so they always match your run.

In [ ]:
best_ms = results["RMSE"].idxmin()
best_wf = wf_results["RMSE"].idxmin()
beaters = [m_ for m_ in results.index if results.loc[m_, "Beats Naive?"] == "Yes"]
print("KEY FINDINGS")
print("-" * 70)
print(f"1. Prices are non-stationary; log-returns are stationary (d = 1). ARIMA order chosen: {best_order}.")
print(f"2. Returns are not normal (excess kurtosis {stats.kurtosis(r):.2f}) and show volatility clustering "
      f"(Ljung-Box p on squared returns = {ljung_box(r ** 2, 10)[1]:.4f}).")
print(f"3. 60-day hold-out: best model by RMSE = {best_ms} (RMSE {results.loc[best_ms, 'RMSE']:.2f}, "
      f"Naive {results.loc['Naive', 'RMSE']:.2f}).")
print("   Models beating Naive on both MAE and RMSE:", beaters if beaters else "NONE")
print(f"4. Walk-forward one-step: best model = {best_wf} (RMSE {wf_results.loc[best_wf, 'RMSE']:.2f}, "
      f"Naive {wf_results.loc['Naive', 'RMSE']:.2f}; difference vs Naive "
      f"{wf_results.loc[best_wf, 'RMSE vs Naive (%)']:+.1f}%).")
print(f"5. 95% interval coverage on the hold-out: " + ", ".join(f"{k} {v:.0f}%" for k, v in cov.items()))
print("6. Residuals:", "structure remains" if p1 < 0.05 else "no leftover autocorrelation",
      "| squared residuals:", "ARCH effect present" if p2 < 0.05 else "no ARCH effect")

### Limitations — why forecasting markets is hard
* **Prices behave close to a random walk.** If today's price already reflects all known information, the best guess for tomorrow is *today's price* — which is why the Naive model is so hard to beat.
* **Errors grow with the horizon.** Forecast uncertainty compounds, so the 95% interval widens quickly. Point forecasts far ahead are almost meaningless without their interval.
* **Volatility is not constant.** ARIMA, SARIMA and Prophet assume constant variance, but returns show volatility clustering and fat tails, so their intervals can be too narrow or too wide. A GARCH model is the natural next step.
* **Shocks are unpredictable.** Crashes and sudden rallies (the injected events in this data) cannot be forecast from past prices alone.
* **Seasonality is weak.** A yearly pattern is small compared to daily noise, so Prophet's seasonal component adds little.
* **The data is synthetic.** Results illustrate the *method*; real equities have news, earnings, liquidity and regime changes.
* **Single hold-out window.** Sixty days is a small test sample; conclusions could change on another window (use longer walk-forward back-tests in practice).

### Possible extensions
GARCH volatility models, adding exogenous features (volume, technical indicators), machine-learning models (XGBoost, LSTM) on lagged returns, and trading-strategy back-tests that include transaction costs.